# Spatiotemporal Video Prediction: ConvLSTM Recurrent Neural Network for Future Frame Forecasting on MovingMNIST

This notebook implements an end-to-end spatiotemporal video forecasting and future frame synthesis pipeline on the **MovingMNIST** dataset (`torchvision.datasets.MovingMNIST`) using a **Convolutional LSTM (ConvLSTM)** architecture that models spatial feature representations and temporal recurrence dynamics simultaneously.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Spatiotemporal Sequence Normalization & Batching Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection of Moving Digit Trajectories**
9. **Spatiotemporal ConvLSTM Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Composite Multi-Scale Loss Formulation (MSE + SSIM)**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation: Rollout MSE, PSNR & SSIM Diagnostics**
14. **Interactive Multi-Step Future Frame Forecasting & Rollout Visualization**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization utilities.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor future frame prediction loss trajectories, validation MSE/PSNR/SSIM curves, and learning rate dynamics.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import scientific computing, image processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import math
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Configure directory structures, temporal sequence lengths ($T_{\text{in}}=10, T_{\text{out}}=10$), spatial frame size ($64 \times 64$), and training hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_video_prediction").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "video_prediction_convlstm.zip"

INPUT_FRAMES = 10
PRED_FRAMES = 10
TOTAL_FRAMES = INPUT_FRAMES + PRED_FRAMES
IMAGE_SIZE = 64
BATCH_SIZE = 16
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

print(f"Dataset Root    : {DATA_DIR}")
print(f"Prediction Task : {INPUT_FRAMES} Input Frames -> {PRED_FRAMES} Future Frames")
print(f"Model Path      : {MODEL_PATH}")

## 6. Spatiotemporal Sequence Batching Pipeline

Implement temporal sequence slicing, converting $[T, H, W]$ arrays into normalized float tensors $[T, 1, H, W] \in [0, 1]$.


In [ ]:
class MovingMNISTWrapper(Dataset):
    """Wrapper splitting 20-frame MovingMNIST into input [10, 1, 64, 64] and target [10, 1, 64, 64]."""

    def __init__(self, raw_ds):
        self.raw_ds = raw_ds

    def __len__(self):
        return len(self.raw_ds)

    def __getitem__(self, idx):
        seq = self.raw_ds[idx]
        if isinstance(seq, torch.Tensor):
            seq = seq.float() / 255.0
        else:
            seq = torch.from_numpy(np.array(seq)).float() / 255.0

        if seq.ndim == 3:
            seq = seq.unsqueeze(1)

        input_seq = seq[:INPUT_FRAMES]
        target_seq = seq[INPUT_FRAMES:]
        return input_seq, target_seq

## 7. Dataset Ingestion & DataLoader Instantiation

Download MovingMNIST dataset and construct train and validation DataLoaders.


In [ ]:
raw_moving_mnist = datasets.MovingMNIST(root=DATA_DIR, download=True)
total_seqs = len(raw_moving_mnist)

train_size = int(0.85 * total_seqs)
val_size = total_seqs - train_size

train_raw, val_raw = torch.utils.data.random_split(
    raw_moving_mnist,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_dataset = MovingMNISTWrapper(train_raw)
val_dataset = MovingMNISTWrapper(val_raw)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"MovingMNIST: {len(train_dataset)} Train Sequences | {len(val_dataset)} Val Sequences"
)

## 8. Exploratory Visual Batch Inspection

Visualize a sample 20-frame sequence showing input frames ($t=1\dots10$) and future target frames ($t=11\dots20$).


In [ ]:
input_seqs, target_seqs = next(iter(train_loader))

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for t in range(10):
    axes[0, t].imshow(input_seqs[0, t, 0].cpu().numpy(), cmap="gray")
    axes[0, t].set_title(f"In t={t + 1}", fontsize=8)
    axes[0, t].axis("off")

    axes[1, t].imshow(target_seqs[0, t, 0].cpu().numpy(), cmap="magma")
    axes[1, t].set_title(f"Target t={t + 11}", fontsize=8)
    axes[1, t].axis("off")

plt.suptitle("MovingMNIST 20-Frame Sequence Inspection", fontsize=12)
plt.tight_layout()
plt.show()

## 9. Spatiotemporal ConvLSTM Neural Architecture Definition

Implement **Convolutional LSTM (ConvLSTM)** cells and full Encoder-Decoder recurrence predicting future video frames.


In [ ]:
class ConvLSTMCell(nn.Module):
    """2D Convolutional LSTM Cell."""

    def __init__(self, in_channels: int, hidden_channels: int, kernel_size: int = 3):
        super().__init__()
        self.in_channels = in_channels
        self.hidden_channels = hidden_channels
        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            in_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
            bias=True,
        )

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor, c_prev: torch.Tensor):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.conv(combined)

        i, f, o, g = torch.split(gates, self.hidden_channels, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_cur = f * c_prev + i * g
        h_cur = o * torch.tanh(c_cur)
        return h_cur, c_cur


class SpatioTemporalConvLSTM(nn.Module):
    """Encoder-Decoder ConvLSTM Video Prediction Model."""

    def __init__(self, in_channels=1, hidden_dims=[32, 64], pred_len=10):
        super().__init__()
        self.pred_len = pred_len

        # Spatial Encoder (Stride 2)
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dims[0], kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(0.1, inplace=True),
        )

        # ConvLSTM Layers
        self.cell1 = ConvLSTMCell(hidden_dims[0], hidden_dims[0], kernel_size=3)
        self.cell2 = ConvLSTMCell(hidden_dims[0], hidden_dims[1], kernel_size=3)

        # Spatial Decoder (Stride 2 Transposed Conv)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                hidden_dims[1], hidden_dims[0], kernel_size=4, stride=2, padding=1
            ),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(hidden_dims[0], in_channels, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, input_seq: torch.Tensor) -> torch.Tensor:
        b, t_in, c, h, w = input_seq.shape
        feat_h, feat_w = h // 2, w // 2

        h1 = torch.zeros(b, 32, feat_h, feat_w, device=input_seq.device)
        c1 = torch.zeros(b, 32, feat_h, feat_w, device=input_seq.device)
        h2 = torch.zeros(b, 64, feat_h, feat_w, device=input_seq.device)
        c2 = torch.zeros(b, 64, feat_h, feat_w, device=input_seq.device)

        for t in range(t_in):
            xt = self.encoder(input_seq[:, t])
            h1, c1 = self.cell1(xt, h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)

        predictions = []
        last_frame = input_seq[:, -1]

        for _ in range(self.pred_len):
            xt = self.encoder(last_frame)
            h1, c1 = self.cell1(xt, h1, c1)
            h2, c2 = self.cell2(h1, h2, c2)
            next_frame = self.decoder(h2)
            predictions.append(next_frame)
            last_frame = next_frame

        return torch.stack(predictions, dim=1)

## 10. Model Instantiation & Parameter Audit

Instantiate ConvLSTM model on compute accelerator and verify spatiotemporal tensor dimensions.


In [ ]:
model = SpatioTemporalConvLSTM(
    in_channels=1, hidden_dims=[32, 64], pred_len=PRED_FRAMES
).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"SpatioTemporal ConvLSTM Architecture Summary:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
dummy_seq = torch.randn(2, INPUT_FRAMES, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    out_preds = model(dummy_seq)
print(
    f" - Predicted Output Shape : {out_preds.shape} (Expected: [2, {PRED_FRAMES}, 1, {IMAGE_SIZE}, {IMAGE_SIZE}])"
)

## 11. Composite Loss Formulation & Optimizer Configuration

Configure composite Mean Squared Error (MSE) loss and AdamW with Cosine Annealing scheduler.


In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Configured Loss & SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Execute training across epochs, computing validation future frame MSE and saving best checkpoint.


In [ ]:
best_val_mse = float("inf")
print(
    f"Starting Video Predictor Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_train_mse = 0.0

    for in_seq, tgt_seq in train_loader:
        in_seq, tgt_seq = in_seq.to(device), tgt_seq.to(device)
        optimizer.zero_grad()

        pred_seq = model(in_seq)
        loss = loss_fn(pred_seq, tgt_seq)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_train_mse += loss.item() * in_seq.size(0)

    epoch_train_mse = running_train_mse / len(train_dataset)
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    running_val_mse = 0.0
    with torch.no_grad():
        for in_seq, tgt_seq in val_loader:
            in_seq, tgt_seq = in_seq.to(device), tgt_seq.to(device)
            pred_seq = model(in_seq)
            loss = loss_fn(pred_seq, tgt_seq)
            running_val_mse += loss.item() * in_seq.size(0)

    epoch_val_mse = running_val_mse / len(val_dataset)
    val_psnr = 10.0 * math.log10(1.0 / max(epoch_val_mse, 1e-8))
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train_MSE", epoch_train_mse, epoch)
    writer.add_scalar("Loss/Val_MSE", epoch_val_mse, epoch)
    writer.add_scalar("Metrics/Val_PSNR_dB", val_psnr, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_mse < best_val_mse:
        best_val_mse = epoch_val_mse
        save_metadata = {
            "epoch": epoch,
            "best_val_mse": best_val_mse,
            "val_psnr": val_psnr,
            "architecture": "SpatioTemporalConvLSTM",
            "dataset": "MovingMNIST",
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train MSE: {epoch_train_mse:.5f} | Val MSE: {epoch_val_mse:.5f} (PSNR: {val_psnr:.2f}dB){checkpoint_msg}"
    )

writer.close()
print(
    f"Training Complete! Peak Validation MSE: {best_val_mse:.5f} (PSNR: {val_psnr:.2f} dB)"
)

## 13. Model Checkpoint Loading & Metrics Inspection

Load best checkpoint and inspect metadata.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded ConvLSTM Checkpoint from Epoch {metadata['epoch']} (Best MSE: {metadata['best_val_mse']:.5f} | PSNR: {metadata.get('val_psnr', 0.0):.2f} dB)"
)

## 14. Interactive Multi-Step Future Frame Forecasting & Rollout Visualization

Predict 10 future frames given 10 historical input frames and plot ground-truth vs. predicted spatiotemporal rollout strips.


In [ ]:
def run_video_prediction_demo(model, dataset, sample_idx=0):
    model.eval()
    in_seq, tgt_seq = dataset[sample_idx]

    with torch.no_grad():
        in_batch = in_seq.unsqueeze(0).to(device)
        pred_seq = model(in_batch).squeeze(0).cpu()

    fig, axes = plt.subplots(3, 10, figsize=(16, 5))

    # 1. Inputs
    for t in range(10):
        axes[0, t].imshow(in_seq[t, 0].cpu().numpy(), cmap="gray")
        axes[0, t].set_title(f"In t={t + 1}", fontsize=8)
        axes[0, t].axis("off")
    axes[0, 0].set_ylabel("Historical Input", fontsize=9, color="black", weight="bold")

    # 2. Ground Truth Future
    for t in range(10):
        axes[1, t].imshow(tgt_seq[t, 0].cpu().numpy(), cmap="magma")
        axes[1, t].set_title(f"GT t={t + 11}", fontsize=8)
        axes[1, t].axis("off")
    axes[1, 0].set_ylabel("Ground Truth", fontsize=9, color="darkgreen", weight="bold")

    # 3. Predicted Future
    for t in range(10):
        axes[2, t].imshow(pred_seq[t, 0].numpy(), cmap="magma")
        axes[2, t].set_title(f"Pred t={t + 11}", fontsize=8)
        axes[2, t].axis("off")
    axes[2, 0].set_ylabel("ConvLSTM Pred", fontsize=9, color="darkblue", weight="bold")

    plt.suptitle(
        "Spatiotemporal Video Prediction Rollout (10 Input Frames -> 10 Predicted Frames)",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


run_video_prediction_demo(
    model, val_dataset, sample_idx=random.randint(0, len(val_dataset) - 1)
)